In [6]:
!pip install git+https://github.com/openai/CLIP.git -q
!pip install torch Pillow numpy matplotlib scikit-learn imageio requests -q

  Preparing metadata (setup.py) ... done


## **Taller 48 - Embeddings Visuales: Proyectando Significados con CLIP y PCA**

In [7]:
import clip
import torch
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import imageio.v2 as imageio
import requests
import urllib.request
import json
from io import BytesIO

###El objetivo del taller es implementar la visualización de relaciones semánticas entre imágenes utilizando embeddings generados por el modelo CLIP de OpenAI y técnicas de reducción de dimensionalidad como PCA y t-SNE.

In [8]:
# --------------------------------------
# Configuración inicial
# --------------------------------------
# Selecciona el dispositivo (GPU si está disponible, sino CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

# Carga el modelo CLIP (ViT-B/32) y su función de preprocesamiento
model, preprocess = clip.load("ViT-B/32", device=device)

# --------------------------------------
# Configuración de la Unsplash API
# --------------------------------------
# Clave de acceso a la API de Unsplash
UNSPLASH_ACCESS_KEY = "zumHjJIW21Euw7OkBEtOfCZiXdHZkt21t4SowsWdZxg"
UNSPLASH_API_URL = "https://api.unsplash.com/search/photos"
HEADERS = {"Authorization": f"Client-ID {UNSPLASH_ACCESS_KEY}"}

# --------------------------------------
# Función para obtener imágenes de Unsplash
# --------------------------------------
def fetch_unsplash_images(query, count=5, save_dir="/content/images"):
    """
    Obtiene imágenes de Unsplash usando la API.
    Args:
        query (str): Término de búsqueda (e.g., "cat").
        count (int): Número de imágenes a descargar.
        save_dir (str): Directorio donde se guardarán las imágenes.
    Returns:
        list: Lista de rutas de las imágenes descargadas.
    """
    os.makedirs(save_dir, exist_ok=True)
    image_paths = []
    try:
        # Realiza la solicitud a la API
        response = requests.get(
            UNSPLASH_API_URL,
            headers=HEADERS,
            params={"query": query, "per_page": count}
        )
        if response.status_code != 200:
            raise Exception(f"Error en la API de Unsplash: {response.status_code} - {response.text}")

        # Procesa las imágenes obtenidas
        data = response.json()
        for i, photo in enumerate(data["results"]):
            url = photo["urls"]["regular"]
            photographer = photo["user"]["name"]
            image_path = os.path.join(save_dir, f"{query}_{i}.jpg")
            urllib.request.urlretrieve(url, image_path)
            image_paths.append(image_path)
            # Guarda la atribución requerida por Unsplash
            with open(os.path.join(save_dir, f"{query}_{i}_attribution.txt"), "w") as f:
                f.write(f"Foto por {photographer} en Unsplash\n{photo['links']['html']}")
        return image_paths
    except Exception as e:
        print(f"Error al obtener imágenes para '{query}': {e}")
        return []

# --------------------------------------
# Obtención de imágenes
# --------------------------------------
# Términos de búsqueda para las imágenes
queries = ["cat", "dog", "tree"]
image_paths = []
for query in queries:
    image_paths.extend(fetch_unsplash_images(query, count=5))

# Verifica que se hayan obtenido imágenes
if not image_paths:
    raise FileNotFoundError("No se obtuvieron imágenes de Unsplash. Verifica la clave API o las consultas.")

# --------------------------------------
# Preprocesamiento de imágenes
# --------------------------------------
images = []
for path in image_paths:
    try:
        # Carga y preprocesa cada imagen para CLIP
        img = preprocess(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
        images.append(img)
    except Exception as e:
        print(f"Error al procesar la imagen {path}: {e}")

# --------------------------------------
# Generación de embeddings con CLIP
# --------------------------------------
# Genera embeddings de 512 dimensiones para las imágenes
with torch.no_grad():
    image_features = [model.encode_image(img).cpu().numpy() for img in images]
X = np.vstack(image_features)  # Combina los embeddings en una matriz

# --------------------------------------
# Reducción de dimensionalidad
# --------------------------------------
# PCA: Reduce los embeddings a 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# t-SNE: Reduce los embeddings a 2D (perplexity ajustado al número de imágenes)
tsne = TSNE(n_components=2, perplexity=min(30, len(image_paths)-1), random_state=42)
X_tsne = tsne.fit_transform(X)

# --------------------------------------
# Clustering con KMeans
# --------------------------------------
# Aplica KMeans para agrupar los embeddings en 3 clústeres
kmeans = KMeans(n_clusters=3, random_state=42)
cluster_labels = kmeans.fit_predict(X)

# --------------------------------------
# Creación de directorios para resultados
# --------------------------------------
os.makedirs("/content/graficos", exist_ok=True)

# --------------------------------------
# Visualización: PCA con miniaturas y clústeres
# --------------------------------------
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis', s=100)
for i, path in enumerate(image_paths):
    img = Image.open(path).resize((32, 32))
    imagebox = plt.matplotlib.offsetbox.OffsetImage(img, zoom=1)
    ab = plt.matplotlib.offsetbox.AnnotationBbox(imagebox, X_pca[i])
    plt.gca().add_artist(ab)
plt.title("Proyección PCA de Embeddings de Imágenes con Clustering KMeans")
plt.xlabel("Componente PCA 1")
plt.ylabel("Componente PCA 2")
plt.colorbar(scatter, label='Clúster')
plt.savefig("/content/graficos/pca_projection.png")
plt.close()

# --------------------------------------
# Visualización: t-SNE con miniaturas y clústeres
# --------------------------------------
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=cluster_labels, cmap='viridis', s=100)
for i, path in enumerate(image_paths):
    img = Image.open(path).resize((32, 32))
    imagebox = plt.matplotlib.offsetbox.OffsetImage(img, zoom=1)
    ab = plt.matplotlib.offsetbox.AnnotationBbox(imagebox, X_tsne[i])
    plt.gca().add_artist(ab)
plt.title("Proyección t-SNE de Embeddings de Imágenes con Clustering KMeans")
plt.xlabel("Componente t-SNE 1")
plt.ylabel("Componente t-SNE 2")
plt.colorbar(scatter, label='Clúster')
plt.savefig("/content/graficos/tsne_projection.png")
plt.close()


Usando dispositivo: cpu


In [9]:
# --------------------------------------
# Bonus: Embeddings de texto
# --------------------------------------
# Genera embeddings para prompts de texto
prompts = ["cat", "dog", "tree"]
text_inputs = torch.cat([clip.tokenize(p).to(device) for p in prompts])
with torch.no_grad():
    text_features = model.encode_text(text_inputs).cpu().numpy()

# Combina embeddings de imágenes y texto
X_combined = np.vstack([X, text_features])
X_pca_combined = pca.fit_transform(X_combined)

# Visualización: Imágenes y texto combinados
plt.figure(figsize=(10, 8))
plt.scatter(X_pca_combined[:len(image_paths), 0], X_pca_combined[:len(image_paths), 1], c='blue', label='Imágenes', s=100)
plt.scatter(X_pca_combined[len(image_paths):, 0], X_pca_combined[len(image_paths):, 1], c='red', label='Prompts de Texto', s=150, marker='x')
for i, path in enumerate(image_paths):
    img = Image.open(path).resize((32, 32))
    imagebox = plt.matplotlib.offsetbox.OffsetImage(img, zoom=1)
    ab = plt.matplotlib.offsetbox.AnnotationBbox(imagebox, X_pca_combined[i])
    plt.gca().add_artist(ab)
for i, prompt in enumerate(prompts):
    plt.text(X_pca_combined[len(image_paths)+i, 0], X_pca_combined[len(image_paths)+i, 1], prompt, fontsize=12, color='red')
plt.title("Proyección PCA de Embeddings de Imágenes y Texto")
plt.xlabel("Componente PCA 1")
plt.ylabel("Componente PCA 2")
plt.legend()
plt.savefig("/content/graficos/combined_projection.png")
plt.close()


In [18]:
# --------------------------------------
# GIF: Transition from t-SNE to PCA Projection
# --------------------------------------
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import numpy as np
from sklearn.cluster import KMeans
from PIL import Image

# Ensure KMeans centroids are available and transformed correctly
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(X)
cluster_labels = kmeans.labels_

# Transform centroids using the fitted PCA (both start and end with PCA-transformed centroids)
centroids_pca = pca.transform(kmeans.cluster_centers_)
centroids_tsne = pca.transform(kmeans.cluster_centers_)  # Use PCA centroids as a starting approximation for consistency

# Determine the overall limits for the plot to ensure consistent frame size
all_points = np.vstack([X_pca, X_tsne])
x_min, x_max = all_points[:, 0].min() - 1, all_points[:, 0].max() + 1
y_min, y_max = all_points[:, 1].min() - 1, all_points[:, 1].max() + 1

# Define a fixed size for the GIF frames
gif_frame_size = (800, 600)

# Generate frames for the GIF
images_for_gif = []
n_transition_frames = 20  # Number of frames for the transition
n_pca_frames = 15        # Increased number of frames to hold the final PCA state

for i in range(n_transition_frames):
    plt.figure(figsize=(10, 8))

    # Interpolate between t-SNE and PCA coordinates with a non-linear transition to emphasize the end
    alpha = i / (n_transition_frames - 1)  # Linear interpolation factor (0 = t-SNE, 1 = PCA)
    alpha_smooth = alpha ** 2  # Non-linear interpolation to slow down near the end
    X_interpolated = (1 - alpha_smooth) * X_tsne + alpha_smooth * X_pca
    centroids_interpolated = (1 - alpha_smooth) * centroids_tsne + alpha_smooth * centroids_pca

    # Plot interpolated points with cluster colors
    scatter = plt.scatter(X_interpolated[:, 0], X_interpolated[:, 1], c=cluster_labels, cmap='viridis', s=100, alpha=0.8)

    # Add image thumbnails
    for j, path in enumerate(image_paths):
        try:
            img = Image.open(path).resize((32, 32))
            imagebox = plt.matplotlib.offsetbox.OffsetImage(img, zoom=1)
            ab = plt.matplotlib.offsetbox.AnnotationBbox(imagebox, X_interpolated[j])
            plt.gca().add_artist(ab)
        except Exception as e:
            print(f"Error adding image {path} to plot: {e}")

    # Plot interpolated centroids
    plt.scatter(centroids_interpolated[:, 0], centroids_interpolated[:, 1], c='red', marker='x', s=200, linewidths=3, label='Centroids')

    # Customize plot
    plt.title(f"t-SNE to PCA Transition: Frame {i+1}/{n_transition_frames}")
    plt.xlabel("Componente PCA 1")
    plt.ylabel("Componente PCA 2")
    plt.colorbar(scatter, label='Clúster')
    plt.legend()
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)

    # Save frame with bbox_inches='tight'
    frame_path = f"/content/graficos/frame_{i}.png"
    plt.savefig(frame_path, bbox_inches='tight')
    img_frame = Image.open(frame_path).resize(gif_frame_size)
    images_for_gif.append(np.array(img_frame))
    plt.close()

# Add extra frames showing the final PCA state
for i in range(n_pca_frames):
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis', s=100, alpha=0.8)
    for j, path in enumerate(image_paths):
        try:
            img = Image.open(path).resize((32, 32))
            imagebox = plt.matplotlib.offsetbox.OffsetImage(img, zoom=1)
            ab = plt.matplotlib.offsetbox.AnnotationBbox(imagebox, X_pca[j])
            plt.gca().add_artist(ab)
        except Exception as e:
            print(f"Error adding image {path} to plot: {e}")
    plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], c='red', marker='x', s=200, linewidths=3, label='Centroids')
    plt.title("Proyección PCA Final con Clustering KMeans")
    plt.xlabel("Componente PCA 1")
    plt.ylabel("Componente PCA 2")
    plt.colorbar(scatter, label='Clúster')
    plt.legend()
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)

    frame_path = f"/content/graficos/frame_{n_transition_frames + i}.png"
    plt.savefig(frame_path, bbox_inches='tight')
    img_frame = Image.open(frame_path).resize(gif_frame_size)
    images_for_gif.append(np.array(img_frame))
    plt.close()

# Create the GIF
imageio.mimsave("/content/graficos/tsne_to_pca_transition.gif", images_for_gif, fps=5)

# Clean up frame files
for i in range(n_transition_frames + n_pca_frames):
    os.remove(f"/content/graficos/frame_{i}.png")